# Firefly: the most massive BRAHMA galaxy and its wandering black holes

Builds a Firefly scene for one BRAHMA subhalo in which

* **gas** and **stars** are separate particle groups,
* the **central black holes** — *every* black hole inside the stellar
  half-mass radius — are split by **mass decade**, so the nuclear region does
  not collapse to a single point and the un-grown seeds sitting there are
  visibly distinct from the SMBHs, and
* the **wandering black holes** are one group in a single contrasting colour,
  so the halo reads as one population against the nuclear one.

The later sections attach a camera fly-through (`TweenParams`), set Firefly's
video-capture defaults, and give the `ffmpeg` command to turn the captured
frames into a movie.

Wandering/central follows the definition already used in
`plotly_interactive_wanderers.ipynb`: a black hole is central when it sits
inside its subhalo's stellar half-mass radius.

## 0. Configuration

Everything you would normally want to change for a different galaxy, a
different snapshot, or a tighter framing lives in this one cell.

In [ ]:
import os

import numpy as np
import h5py

from cosmo_sim_tools import brahma
from cosmo_sim_tools.brahma.snapshot import snapPath_groupordered, loadSubset_groupordered
from cosmo_sim_tools.arepo_tools import mdot_to_Lbol

BASEPATH = ('/orange/lblecha/aklantbhowmick/GAS_BASED_SEED_MODEL_UNIFORM_RUNS/'
            'L12p5n512/AREPO/SM5_DFD_3_TNG')
SNAP = 25                      # 32 -> z = 0, 25 -> z = 1.0

# None -> the most massive galaxy (by stellar mass) in this snapshot.
SUBHALO_INDEX = 683

# Scene half-width in physical kpc from the galaxy centre. This is a framing
# choice, not a physical one, and it has to track the galaxy: the wandering BHs
# of the z=0 central galaxy reach ~1 Mpc and wanted 400 kpc, but subhalo 683 at
# z=1 is 15x less massive and its most distant BH is only 69.7 kpc out, so 80
# kpc holds all 129 of them with a little margin to spare.
SCENE_RADIUS_KPC = 100.0

# Firefly renders in the browser, so cap what we ship. 5e4 per component loads
# in about a second and keeps the browser responsive, which is what you want
# while checking that the GUI builds; the black holes are unaffected either way,
# since they are never subsampled. Raise both to 5e5 for the final render -- that
# is comfortable on a laptop, and the GUI's own decimation slider thins it
# further live.
# This galaxy has only 127,518 gas cells and 129,981 star particles in total,
# so 150,000 ships every one of them -- no subsampling, which is what you want
# for a recording. That is still a quarter of the z=0 central galaxy's load.
MAX_GAS = 150_000
MAX_STARS = 150_000
MAX_DM = 0                     # 0 -> no dark matter group at all

# A BH is "central" if it sits within CENTRAL_RADIUS_FACTOR x the stellar
# half-mass radius. Everything outside that wanders.
CENTRAL_RADIUS_FACTOR = 1.0

RADIATIVE_EFFICIENCY = 0.2     # L_bol = eps Mdot c^2, the codebase default
SEED = 42

DATADIR = os.path.expanduser('~/firefly_brahma_wanderers')

rng = np.random.default_rng(SEED)
print('Firefly dataset will be written to', DATADIR)

## 1. Reading one subhalo without reading the whole box

The postprocessed catalogue orders particles group by group and, inside a
group, subhalo by subhalo, so a subhalo's particles are one contiguous slice of
the group-ordered snapshot. `loadSubset_groupordered` will read only that slice
if it is handed a `subset` dictionary, which is what the helpers below build.

This matters here: `arepo_package.get_particle_property_within_postprocessed_groups`
(used by `bh_2dplot.py` and the older Firefly notebook) loads the *entire box*
for every field — 1.2e8 gas cells at snapshot 32 — and does it again per field.
Slicing instead turns each read into a fraction of a second.

In [ ]:
_OFFSET_CACHE = {}


def _snapshot_file_offsets(basepath, snap):
    """Cumulative count of each particle type at the start of every snapshot chunk."""
    with h5py.File(snapPath_groupordered(basepath, snap, 0), 'r') as handle:
        n_files = int(handle['Header'].attrs['NumFilesPerSnapshot'])

    per_file = np.zeros((6, n_files), dtype=np.int64)
    for chunk in range(n_files):
        with h5py.File(snapPath_groupordered(basepath, snap, chunk), 'r') as handle:
            per_file[:, chunk] = np.asarray(
                handle['Header'].attrs['NumPart_ThisFile'][:6], dtype=np.int64)

    running = np.cumsum(per_file, axis=1)
    return np.concatenate([np.zeros((6, 1), dtype=np.int64), running], axis=1)[:, :-1]


def _subhalo_offsets(basepath, snap):
    """Per-subhalo (offset, length) by particle type, in group-ordered indices."""
    subhalos = brahma.groupcat.loadSubhalos_postprocessed(
        basepath, snap, fields=['SubhaloLenType', 'SubhaloGrNr'])
    groups = brahma.groupcat.loadHalos_postprocessed(
        basepath, snap, fields=['GroupLenType', 'GroupFirstSub'])

    lengths = subhalos['SubhaloLenType'].astype(np.int64)
    group_of = subhalos['SubhaloGrNr'].astype(np.int64)
    group_lengths = groups['GroupLenType'].astype(np.int64)
    first_subhalo = groups['GroupFirstSub'].astype(np.int64)

    def exclusive_cumsum(array):
        return np.concatenate(
            [np.zeros((1, 6), dtype=np.int64), np.cumsum(array, axis=0)], axis=0)[:-1]

    group_start = exclusive_cumsum(group_lengths)
    running = exclusive_cumsum(lengths)
    # Subhalos are themselves stored in group order, so subtracting the running
    # total at the group's first subhalo leaves the offset within the group.
    offsets = group_start[group_of] + running - running[first_subhalo[group_of]]
    return offsets, lengths


def subhalo_subset(basepath, snap, subhalo_index):
    """The `subset` dictionary `loadSubset_groupordered` needs for one subhalo."""
    key = (basepath, snap)
    if key not in _OFFSET_CACHE:
        _OFFSET_CACHE[key] = (_snapshot_file_offsets(basepath, snap),
                              *_subhalo_offsets(basepath, snap))
    snap_offsets, offsets, lengths = _OFFSET_CACHE[key]
    return {'snapOffsets': snap_offsets,
            'offsetType': offsets[subhalo_index],
            'lenType': lengths[subhalo_index]}


def load_subhalo(basepath, snap, subhalo_index, particle_type, fields):
    """Requested fields for one particle type of one subhalo; None if it has none."""
    subset = subhalo_subset(basepath, snap, subhalo_index)
    if subset['lenType'][particle_type] == 0:
        return None
    return loadSubset_groupordered(basepath, snap, particle_type, fields=list(fields),
                                   subset=subset, sq=False, float32=True)

## 2. Pick the galaxy

The header gives the unit conversions; the catalogue gives the centre and the
stellar half-mass radius that the central/wandering split is measured against.

In [ ]:
header = brahma.groupcat.loadHeader(BASEPATH, SNAP)
hubble_param = float(header['HubbleParam'])
boxsize = float(header['BoxSize'])              # ckpc/h
scale_factor = float(header['Time'])
redshift = float(header['Redshift'])

catalogue = brahma.groupcat.loadSubhalos_postprocessed(
    BASEPATH, SNAP,
    fields=['SubhaloPos', 'SubhaloMassType', 'SubhaloLenType', 'SubhaloHalfmassRadType'])

stellar_mass = catalogue['SubhaloMassType'][:, 4] * 1e10 / hubble_param
ranking = np.argsort(stellar_mass)[::-1]

print(f'snapshot {SNAP}:  z = {redshift:.3f}   a = {scale_factor:.4f}   h = {hubble_param:.4f}')
print('\nmost massive galaxies in this snapshot')
print(f'{"subhalo":>8} {"Mstar [Msun]":>14} {"N_BH":>6} {"N_gas":>10} {"N_star":>10} {"r_1/2,* [kpc]":>14}')
for index in ranking[:5]:
    print(f'{index:8d} {stellar_mass[index]:14.3e} '
          f'{catalogue["SubhaloLenType"][index, 5]:6d} '
          f'{catalogue["SubhaloLenType"][index, 0]:10d} '
          f'{catalogue["SubhaloLenType"][index, 4]:10d} '
          f'{catalogue["SubhaloHalfmassRadType"][index, 4] * scale_factor / hubble_param:14.2f}')

subhalo_index = int(ranking[0]) if SUBHALO_INDEX is None else int(SUBHALO_INDEX)
centre = np.asarray(catalogue['SubhaloPos'][subhalo_index], dtype=np.float64)
half_mass_radius = float(
    catalogue['SubhaloHalfmassRadType'][subhalo_index, 4]) * scale_factor / hubble_param

print(f'\ntarget: subhalo {subhalo_index}, '
      f'Mstar = {stellar_mass[subhalo_index]:.3e} Msun, '
      f'r_1/2,* = {half_mass_radius:.2f} physical kpc')


def to_physical_kpc(coordinates):
    """Centre on the subhalo, unwrap the periodic box, ckpc/h -> physical kpc."""
    offset = np.asarray(coordinates, dtype=np.float64) - centre
    offset -= boxsize * np.round(offset / boxsize)
    return offset * scale_factor / hubble_param

## 3. Black holes: which one is central, which ones wander

Every black hole inside the stellar half-mass radius counts as central, and
everything outside `r_1/2,*` wanders. That split is a diagnostic, not a drawing
decision: all of them ship as one flat-coloured swarm whose mass you colour by
picking a field in Firefly's Colormap pane.

The one object drawn apart is the **most massive central black hole**, which
goes into a `BH_central` group of its own so it gets its own colour picker and
size slider in the Particles pane.

The cell prints the three candidates for "the central black hole" — the
innermost, the most massive, and the one at the minimum of the potential — plus
`Time_Of_Seeding` and `BH_Progs` for each, because in this run they need not be
the same object. That divergence is the effect the wandering-BH work is about,
and it is worth reading before trusting any one of them as *the* nucleus. Only
the most massive is split out; the line that picks it says how to choose
another.


In [ ]:
black_holes = load_subhalo(
    BASEPATH, SNAP, subhalo_index, 5,
    ['Coordinates', 'BH_Mass', 'BH_Mdot', 'BH_MdotEddington', 'BH_Progs',
     'Time_Of_Seeding', 'Potential'])
if black_holes is None:
    raise RuntimeError(f'subhalo {subhalo_index} has no black holes')

bh_position = to_physical_kpc(black_holes['Coordinates'])
bh_radius = np.linalg.norm(bh_position, axis=1)                          # physical kpc
bh_mass = np.ravel(black_holes['BH_Mass']).astype(np.float64) * 1e10 / hubble_param
bh_progenitors = np.ravel(black_holes['BH_Progs']).astype(np.int64)
bh_potential = np.ravel(black_holes['Potential']).astype(np.float64)

# Time_Of_Seeding is the scale factor at which the seed was planted, so it says
# whether a black hole still at the seed mass was placed recently or has simply
# been sitting there un-grown for most of the age of the universe.
bh_seed_scale = np.ravel(black_holes['Time_Of_Seeding']).astype(np.float64)
bh_seed_redshift = 1.0 / bh_seed_scale - 1.0

accretion_rate = np.ravel(black_holes['BH_Mdot']).astype(np.float64)
eddington_rate = np.ravel(black_holes['BH_MdotEddington']).astype(np.float64)
bh_luminosity = accretion_rate * mdot_to_Lbol.get_conversion_factor_arepo(
    RADIATIVE_EFFICIENCY)                                                # erg/s
bh_eddington_ratio = np.divide(accretion_rate, eddington_rate,
                               out=np.zeros_like(accretion_rate),
                               where=eddington_rate > 0)

inside_half_mass = bh_radius < CENTRAL_RADIUS_FACTOR * half_mass_radius
if not inside_half_mass.any():
    raise RuntimeError('no black hole inside the stellar half-mass radius; '
                       'raise CENTRAL_RADIUS_FACTOR')

# Central is a region, not a single object: every BH inside r_1/2,*.
is_central = inside_half_mass
is_wandering = ~is_central

in_scene = bh_radius <= SCENE_RADIUS_KPC
wandering_selection = is_wandering & in_scene

# The swarm goes into one group in one colour; mass is left to a colormap chosen
# live in Firefly's Colormap pane, which is why the tracked fields below matter
# more than group membership. The one exception is the most massive black hole
# inside r_1/2,*, split off below into a group of its own so that its colour and
# size can be set independently in the Particles pane. The rest of the
# central/wandering split stays a printed diagnostic, not a drawing decision.
# Every black hole is sent, with no radius cut. There are only ~1e2 of them, so
# they cost nothing beside the ~2.5e5 gas and star points, and cutting them at
# SCENE_RADIUS_KPC discarded the widest wanderers -- exactly the population this
# is built to show. SCENE_RADIUS_KPC now frames gas and stars only.
in_scene = bh_radius <= SCENE_RADIUS_KPC            # reported below, not applied

print(f'{bh_mass.size:,} black holes in subhalo {subhalo_index}: '
      f'{is_central.sum():,} central (inside r_1/2,* = {half_mass_radius:.2f} kpc), '
      f'{is_wandering.sum():,} wandering')
print(f'radii span {bh_radius.min():.2f} - {bh_radius.max():.1f} physical kpc')
print(f'seeded between z = {bh_seed_redshift.max():.2f} and z = {bh_seed_redshift.min():.2f}')

# These three need not be the same black hole, which is the point.
central_indices = np.flatnonzero(is_central)
candidates = [
    ('innermost', int(central_indices[np.argmin(bh_radius[central_indices])])),
    ('most massive', int(central_indices[np.argmax(bh_mass[central_indices])])),
    ('potential minimum', int(central_indices[np.argmin(bh_potential[central_indices])])),
]
print('\ncandidates for "the central black hole"')
print(f'{"":19} {"M_BH [Msun]":>12} {"r [kpc]":>9} {"progs":>6} {"z_seed":>8} {"Potential":>12}')
for label, index in candidates:
    print(f'{label:19} {bh_mass[index]:12.3e} {bh_radius[index]:9.3f} '
          f'{bh_progenitors[index]:6d} {bh_seed_redshift[index]:8.2f} {bh_potential[index]:12.4e}')
if len({index for _, index in candidates}) > 1:
    print('these are NOT the same black hole')

# "Most massive central" is the one that gets its own group. If the three
# candidates above disagree, this is the choice being made -- swap the argmax for
# argmin(bh_radius[...]) or argmin(bh_potential[...]) to pick a different one.
central_bh_index = int(central_indices[np.argmax(bh_mass[central_indices])])
central_bh_selection = np.zeros(bh_mass.size, dtype=bool)
central_bh_selection[central_bh_index] = True
bh_selection = ~central_bh_selection               # everything else, one group

print('\nmass and radius by class')
print(f'{"class":>22} {"N":>7} {"median M_BH":>12} {"min M_BH":>11} {"max M_BH":>11} '
      f'{"median r":>10} {"med z_seed":>11}')
summary = [('central BH (own group)', central_bh_selection),
           ('swarm (one group)', bh_selection),
           ('  of which central', bh_selection & is_central),
           ('  of which wandering', bh_selection & is_wandering),
           ('ALL black holes', np.ones(bh_mass.size, dtype=bool))]
for label, selection in summary:
    count = int(selection.sum())
    if count == 0:
        print(f'{label:>22} {count:7d} {"-":>12} {"-":>11} {"-":>11} {"-":>10} {"-":>11}')
        continue
    mass, radius = bh_mass[selection], bh_radius[selection]
    print(f'{label:>22} {count:7d} {np.median(mass):12.3e} {mass.min():11.3e} '
          f'{mass.max():11.3e} {np.median(radius):10.3f} '
          f'{np.median(bh_seed_redshift[selection]):11.2f}')

print('\nmost massive black holes')
print(f'{"M_BH [Msun]":>12} {"r [kpc]":>9} {"progs":>6} {"z_seed":>8} '
      f'{"L_bol [erg/s]":>14} {"f_Edd":>10}  class')
for index in np.argsort(bh_mass)[::-1][:10]:
    label = ('CENTRAL (own group)' if index == central_bh_index
             else 'central' if is_central[index] else 'wandering')
    print(f'{bh_mass[index]:12.3e} {bh_radius[index]:9.2f} {bh_progenitors[index]:6d} '
          f'{bh_seed_redshift[index]:8.2f} {bh_luminosity[index]:14.3e} '
          f'{bh_eddington_ratio[index]:10.2e}  {label}')

log_mass = np.log10(bh_mass)
print(f'\nsending all {bh_mass.size:,} black holes '
      f'({is_central.sum():,} central, {is_wandering.sum():,} wandering) as two '
      f'groups: the most massive central one alone at '
      f'{bh_mass[central_bh_index]:.3e} Msun, the other {bh_selection.sum():,} '
      f'as the swarm; {(~in_scene).sum():,} of them lie outside the '
      f'{SCENE_RADIUS_KPC:.0f} kpc gas/star cut and are drawn anyway, out to '
      f'{bh_radius.max():.1f} kpc')
print(f'log10 M_BH spans {log_mass.min():.2f} .. {log_mass.max():.2f} '
      f'(median {np.median(log_mass):.2f}) -- the range a Firefly colormap will cover')


## 4. Gas and stars

Both are loaded as separate components, cut to the scene radius, then randomly
subsampled down to the caps set at the top. Derived fields (temperature,
hydrogen number density, stellar age, metallicity) ride along so they can be
used as Firefly colormaps or filters from inside the viewer.

In [ ]:
GAMMA = 5.0 / 3.0
HYDROGEN_FRACTION = 0.76
PROTON_MASS_G = 1.6726219e-24
BOLTZMANN_CGS = 1.380649e-16
SOLAR_MASS_G = 1.989e33
KPC_IN_CM = 3.0857e21
SOLAR_METALLICITY = 0.0127


def load_component(particle_type, fields, max_particles, label):
    """Scene-cut, subsampled particle data for one type, in physical kpc."""
    raw = load_subhalo(BASEPATH, SNAP, subhalo_index, particle_type, fields)
    if raw is None or max_particles == 0:
        print(f'{label}: skipped')
        return None

    position = to_physical_kpc(raw['Coordinates'])
    inside = np.flatnonzero(np.linalg.norm(position, axis=1) <= SCENE_RADIUS_KPC)
    if max_particles is not None and inside.size > max_particles:
        picked = np.sort(rng.choice(inside, size=max_particles, replace=False))
    else:
        picked = inside

    component = {'Coordinates': position[picked]}
    for name in fields:
        if name != 'Coordinates':
            component[name] = np.asarray(raw[name], dtype=np.float64)[picked]

    print(f'{label}: {int(raw["count"]):,} bound -> {inside.size:,} in scene '
          f'-> {picked.size:,} sent to Firefly')
    return component


gas = load_component(0, ['Coordinates', 'Masses', 'Density', 'InternalEnergy',
                         'ElectronAbundance', 'StarFormationRate'], MAX_GAS, 'gas')
stars = load_component(4, ['Coordinates', 'Masses', 'StellarAgeGyr',
                           'GFM_Metallicity'], MAX_STARS, 'stars')
dark_matter = load_component(1, ['Coordinates'], MAX_DM, 'dark matter')

# Gas temperature from the internal energy, the standard Arepo/TNG conversion.
mean_molecular_weight = (4.0 / (1 + 3 * HYDROGEN_FRACTION
                                + 4 * HYDROGEN_FRACTION * gas['ElectronAbundance'])
                         * PROTON_MASS_G)
gas_temperature = ((GAMMA - 1.0) * gas['InternalEnergy'] * 1e10
                   * mean_molecular_weight / BOLTZMANN_CGS)                 # K

# Code density (1e10 Msun/h per (ckpc/h)^3) -> physical hydrogen number density.
gas_density_cgs = (gas['Density'] * 1e10 * hubble_param ** 2 / scale_factor ** 3
                   * SOLAR_MASS_G / KPC_IN_CM ** 3)
gas_hydrogen_density = gas_density_cgs * HYDROGEN_FRACTION / PROTON_MASS_G   # cm^-3

star_metallicity = np.log10(
    np.maximum(stars['GFM_Metallicity'], 1e-8) / SOLAR_METALLICITY)

print(f'\nlog10 T   [K]     {np.log10(gas_temperature).min():6.2f} .. '
      f'{np.log10(gas_temperature).max():6.2f}')
print(f'log10 n_H [cm^-3] {np.log10(gas_hydrogen_density).min():6.2f} .. '
      f'{np.log10(gas_hydrogen_density).max():6.2f}')
print(f'stellar age [Gyr] {stars["StellarAgeGyr"].min():6.2f} .. '
      f'{stars["StellarAgeGyr"].max():6.2f}')

## 5. Firefly particle groups

Each group gets its own colour, size slider and colormap in the viewer, which is
the whole reason the most massive central black hole is a group of its own
rather than one more point in the `BH` swarm.

Two things worth knowing about how Firefly sizes points. The vertex shader
computes `clamp(sizeMult / distance * 2000, minPointScale, maxPointScale)`, so
`maxPointScale` (raised to 25 below) is a hard pixel ceiling on *every* group —
without raising it the central black hole can never draw larger than gas. And
Firefly's per-particle radius scaling normalises the chosen field to [0, 1],
which would shrink the lightest black hole in a group to zero size; that is why
the central black hole is a separate group rather than one radius-scaled group.


In [ ]:
from firefly.data_reader import ParticleGroup, Reader, Settings, TweenParams


def build_group(name, coordinates, colour, size_multiplier,
                fields=None, colormap=None, colormap_field=None):
    """One Firefly particle group, with optional tracked fields and a colormap."""
    group = ParticleGroup(
        name,
        np.asarray(coordinates, dtype=np.float32),
        partsColors=np.asarray(colour, dtype=np.float64),
        partsSizeMultipliers=float(size_multiplier),
        loud=False)

    for field_name, (values, limits) in (fields or {}).items():
        # The filter limits matter as much as the colormap ones: left at None
        # Firefly derives each group's slider range from that group's own data,
        # and a group holding one particle gets a zero-width slider that stalls
        # the GUI build.
        group.trackArray(field_name, np.asarray(values, dtype=np.float32),
                         filterLims=limits, filterVals=limits,
                         colormapLims=limits, colormapVals=limits)

    if colormap_field is not None:
        # Settings.outputToDict turns a colormap *name* into its texture index.
        group.settings_default['colormap'] = colormap
        group.settings_default['colormapVariable'] = colormap_field
        group.settings_default['showColormap'] = True
    return group


groups = [
    build_group(
        'gas', gas['Coordinates'], [0.22, 0.45, 0.92, 1.0], 0.20,
        fields={
            'log10_T_K': (np.log10(gas_temperature), [3.5, 7.5]),
            'log10_nH_cm3': (np.log10(gas_hydrogen_density), [-6.0, -0.5]),
            'SFR_Msun_per_yr': (gas['StarFormationRate'], None),
            'log10_mass_Msun': (np.log10(gas['Masses'] * 1e10 / hubble_param), None),
        },
        # Black-to-blue on density: with additive blending the diffuse halo
        # falls away to nothing and the disk glows, instead of half a million
        # equally bright points fogging over the black holes.
        colormap='BlkBlue', colormap_field='log10_nH_cm3'),
    build_group(
        'stars', stars['Coordinates'], [1.00, 0.88, 0.66, 1.0], 0.10,
        fields={
            'age_Gyr': (stars['StellarAgeGyr'], [0.0, 13.0]),
            'log10_Z_Zsun': (star_metallicity, [-2.0, 0.7]),
            'log10_mass_Msun': (np.log10(stars['Masses'] * 1e10 / hubble_param), None),
        }),
]

if dark_matter is not None:
    groups.append(build_group('DM', dark_matter['Coordinates'],
                              [0.45, 0.45, 0.50, 1.0], 0.08))


def field_limits(values, pad=0.05):
    """Limits for a tracked field, guaranteed to have width.

    Firefly builds one filter slider per field per group, and when the limits
    are None it derives them from that group's own data. A group holding a
    single black hole then gets min == max, a zero-width slider, and the GUI
    build stalls on it -- which is what "stuck on building GUI window" is.
    Deriving the limits from *every* black hole instead keeps each range finite
    and makes a filter mean the same thing in every group.
    """
    low, high = float(np.min(values)), float(np.max(values))
    if not (np.isfinite(low) and np.isfinite(high)):
        return None
    if high - low < 1e-12:
        span = max(abs(high), 1.0) * pad
        return [low - span, high + span]
    return [low, high]


# Round numbers where they read better, the full BH population everywhere else.
BH_FIELD_LIMITS = {
    'log10_MBH_Msun': [5.0, 9.0],
    'radius_kpc': field_limits(bh_radius),
    'log10_Lbol_erg_per_s': [38.0, 46.0],
    'eddington_ratio': field_limits(bh_eddington_ratio),
    'n_progenitors': field_limits(bh_progenitors.astype(float)),
    'z_seeded': field_limits(bh_seed_redshift),
}


def black_hole_fields(selection):
    values = {
        'log10_MBH_Msun': np.log10(bh_mass[selection]),
        'radius_kpc': bh_radius[selection],
        'log10_Lbol_erg_per_s': np.log10(np.maximum(bh_luminosity[selection], 1e30)),
        'eddington_ratio': bh_eddington_ratio[selection],
        'n_progenitors': bh_progenitors[selection].astype(float),
        'z_seeded': bh_seed_redshift[selection],
    }
    return {name: (value, BH_FIELD_LIMITS[name]) for name, value in values.items()}


# One group, one flat colour. Mass is deliberately NOT encoded here: every
# tracked field below is available as a colormap in Firefly's Colormap pane, so
# the choice is made live in the viewer rather than baked in. Orange reads
# clearly against the blue gas and the cream stars if the colormap is off.
BH_COLOUR = [1.00, 0.50, 0.12, 1.0]
BH_SIZE = 4.0                  # the GUI's size slider adjusts this live
# 4.0 rather than 1.0: only 129 black holes here against ~256k gas and star
# points, so at 1.0 everything outside the galaxy core drew a pixel or two wide
# and disappeared into the gas haze.

# The most massive central black hole, alone, so the Particles pane exposes a
# colour picker and a size slider that touch nothing else. BH_FIELD_LIMITS is
# what makes a one-particle group safe: every slider range comes from the whole
# BH population, so none of them collapses to zero width here.
CENTRAL_BH_COLOUR = [0.65, 1.00, 1.00, 1.0]    # cyan-white against the orange swarm
CENTRAL_BH_SIZE = 12.0

groups.append(build_group('BH', bh_position[bh_selection], BH_COLOUR, BH_SIZE,
                          fields=black_hole_fields(bh_selection)))
groups.append(build_group('BH_central', bh_position[central_bh_selection],
                          CENTRAL_BH_COLOUR, CENTRAL_BH_SIZE,
                          fields=black_hole_fields(central_bh_selection)))

for group in groups:
    print(group)


## 6. App settings and the camera path

Firefly tweens camera **position** and **rotation** independently, and the stock
`TweenParams` writes every rotation as `(0, 0, 0)` — a camera that stares down
`-z` no matter where it is. That is fine for a straight dolly and useless for an
orbit, so `LookAtTweenParams` below fills in the three.js `XYZ` Euler angles that
keep the galaxy centred.

Two more quirks are handled here: keyframe durations reach `TWEEN.js`
unconverted and so are **milliseconds**, not the seconds the docstring claims;
and the path is built as a closed loop -- the azimuth swings across
`ORBIT_SWEEP_DEG` and back while the radius eases in to `ORBIT_NEAR_KPC` at the
halfway point -- so that looping playback and the recorded clip join up
seamlessly. `orbit_path` explains why a full 360 degree revolution does not.

In [ ]:
ORBIT_SECONDS = 24             # wall-clock length of one full loop
ORBIT_KEYFRAMES = 240          # camera samples along the loop
ORBIT_FAR_KPC = 2.4 * SCENE_RADIUS_KPC     # widest point: the whole BH swarm
ORBIT_NEAR_KPC = 12.0          # closest point: ~4x r_1/2,*, the galaxy close-up
ORBIT_SWEEP_DEG = 150.0        # azimuth swings +-half of this and back
ORBIT_ELEVATION_DEG = 18.0     # look down on the galaxy by this much
ORBIT_TURNS = None             # set to 1.0 for a full revolution instead


def look_at_euler(positions, target=(0.0, 0.0, 0.0), up=(0.0, 1.0, 0.0)):
    """three.js 'XYZ' Euler angles of a camera at `positions` facing `target`."""
    positions = np.atleast_2d(np.asarray(positions, dtype=np.float64))
    backward = positions - np.asarray(target, dtype=np.float64)
    backward /= np.linalg.norm(backward, axis=1, keepdims=True)

    up = np.asarray(up, dtype=np.float64)
    right = np.cross(np.broadcast_to(up, backward.shape), backward)
    degenerate = np.linalg.norm(right, axis=1) < 1e-8      # camera on the pole
    if degenerate.any():
        nudged = up + np.array([1e-4, 0.0, 1e-4])
        right[degenerate] = np.cross(nudged, backward[degenerate])
    right /= np.linalg.norm(right, axis=1, keepdims=True)
    camera_up = np.cross(backward, right)

    # Matrix columns are (right, camera_up, backward); this is three.js'
    # Euler.setFromRotationMatrix for order 'XYZ', including its gimbal branch.
    angle_y = np.arcsin(np.clip(backward[:, 0], -1.0, 1.0))
    general = np.abs(backward[:, 0]) < 0.9999999
    angle_x = np.where(general,
                       np.arctan2(-backward[:, 1], backward[:, 2]),
                       np.arctan2(camera_up[:, 2], camera_up[:, 1]))
    angle_z = np.where(general, np.arctan2(-camera_up[:, 0], right[:, 0]), 0.0)

    # TWEEN.js interpolates each angle linearly, so keep them continuous across
    # the +-pi branch cut instead of letting a full turn unwind backwards.
    return np.column_stack([np.unwrap(angle_x), np.unwrap(angle_y), np.unwrap(angle_z)])


class LookAtTweenParams(TweenParams):
    """TweenParams whose keyframes also carry a rotation facing `target`."""

    def __init__(self, coords, duration, target=(0.0, 0.0, 0.0),
                 up=(0.0, 1.0, 0.0), **kwargs):
        super().__init__(coords=coords, duration=duration, **kwargs)
        self.target = target
        self.up = up

    def outputToDict(self):
        params = super().outputToDict()
        params['rotation'] = [
            {'x': float(x), 'y': float(y), 'z': float(z)}
            for x, y, z in look_at_euler(self.coordss, self.target, self.up)]
        return params


def orbit_path(n_keyframes, far_kpc, near_kpc=None, sweep_deg=150.0, turns=None,
               elevation_deg=18.0):
    """A closed camera loop around the origin that dives in and pulls back out.

    By default the azimuth swings +-`sweep_deg`/2 and returns, which loops
    seamlessly. Passing `turns` gives a full revolution instead: prettier, but
    one revolution winds the camera's z Euler angle by 2*pi, and because Firefly
    tweens Euler angles it unwinds that in the single keyframe at the loop
    point -- a visible spin once per lap, right where a recording joins up.
    """
    fraction = np.arange(n_keyframes) / n_keyframes
    if turns is None:
        angle = np.radians(sweep_deg / 2.0) * np.sin(2 * np.pi * fraction)
    else:
        angle = 2 * np.pi * turns * fraction

    if near_kpc is None:
        radius = np.full(n_keyframes, float(far_kpc))
    else:
        # sin^2 has zero slope at both ends, so the loop closes without a kink.
        radius = far_kpc * (near_kpc / far_kpc) ** np.sin(np.pi * fraction) ** 2

    elevation = np.radians(elevation_deg)
    return np.column_stack([
        radius * np.cos(elevation) * np.cos(angle),
        radius * np.sin(elevation) * np.ones(n_keyframes),
        radius * np.cos(elevation) * np.sin(angle),
    ])


camera_path = orbit_path(ORBIT_KEYFRAMES, ORBIT_FAR_KPC, near_kpc=ORBIT_NEAR_KPC,
                         sweep_deg=ORBIT_SWEEP_DEG, turns=ORBIT_TURNS,
                         elevation_deg=ORBIT_ELEVATION_DEG)
rotation_path = look_at_euler(camera_path)

# Durations are handed straight to TWEEN.js, which reads milliseconds.
tween_params = LookAtTweenParams(
    camera_path,
    duration=ORBIT_SECONDS * 1000.0 / ORBIT_KEYFRAMES,
    loop=True)

settings = Settings(
    title=f'BRAHMA SM5_DFD_3_TNG - subhalo {subhalo_index} at z = {redshift:.2f}',
    center=[0.0, 0.0, 0.0],
    camera=[float(value) for value in camera_path[0]],
    cameraRotation=[float(value) for value in rotation_path[0]],
    maxPointScale=25,          # pixel ceiling; without this the BHs cannot stand out
    friction=0.15,
    showFPS=False,             # keep overlays out of the recording
    showMemoryUsage=False,
    collapseGUIAtStart=True,
    VideoCapture_FPS=30,
    VideoCapture_duration=int(ORBIT_SECONDS),
    VideoCapture_format=1,     # 0 -> .gif, 1 -> .png frames in a .tar, 2 -> .jpg
    VideoCapture_filename=f'brahma_sub{subhalo_index}_snap{SNAP:03d}',
)

print(f'camera loop: {ORBIT_FAR_KPC:.0f} kpc -> {ORBIT_NEAR_KPC:.0f} kpc -> '
      f'{ORBIT_FAR_KPC:.0f} kpc over {ORBIT_SECONDS} s '
      f'({ORBIT_KEYFRAMES} keyframes, {ORBIT_SECONDS * 1000.0 / ORBIT_KEYFRAMES:.1f} ms each)')
print('rotation wind-up over one loop (rad, should be ~0 for a seamless join): '
      + np.array2string(rotation_path[-1] - rotation_path[0], precision=3))

## 7. Write the dataset

`blendingMode` and `depthTest` are not among the keys `Settings.attachSettings`
copies off a particle group, so they are set on the settings object directly
once every group is attached. Gas and stars get additive blending with the
depth test off, which is what makes them glow; the black holes deliberately do
not, because additive blending makes a point invisible wherever the background
is already bright.

In [ ]:
reader = Reader(datadir=DATADIR, clean_datadir=True, write_startup=True,
                settings=settings)

# Reader's tweenParams check is `__class__.__name__ != 'TweenParams'`, which a
# subclass fails, so the fly-through is attached after construction instead.
reader.tweenParams = tween_params

for group in groups:
    reader.addParticleGroup(group)

# Additive blending with the depth test off is what makes gas and stars glow --
# and it is exactly wrong for the black holes. Additive *adds* a point's colour
# to what is behind it, so over the bright stellar disk, where the channels are
# already near-saturated, a black hole adds nothing visible and disappears;
# only the few sitting on dark halo background survive. Normal blending with the
# depth test on draws them opaque, so they occlude the galaxy and read as
# distinct dots everywhere, which is what the older renders of this galaxy did.
GLOWING_GROUPS = {'gas', 'stars', 'DM'}
settings['blendingMode'] = {
    group.UIname: ('additive' if group.UIname in GLOWING_GROUPS else 'normal')
    for group in groups}
settings['depthTest'] = {
    group.UIname: group.UIname not in GLOWING_GROUPS for group in groups}

reader.writeToDisk()
print(reader)
print('\nwrote', DATADIR)

## 8. Launch the viewer

`spawnFireflyServer` starts a local Flask server; the dataset written above is
picked up from `startup.json`. If the notebook is running on a remote node,
forward the port first (`ssh -L 5500:localhost:5500 <host>`) and open the URL in
a local browser rather than using the inline frame — recording needs a real
browser window anyway.

In [ ]:
from firefly.server import spawnFireflyServer, quitAllFireflyServers

PORT = 5500
server_process = spawnFireflyServer(PORT, method='flask')
viewer_url = f'http://localhost:{PORT:d}/combined'
print('Firefly is running at', viewer_url)

In [ ]:
from IPython.display import IFrame, display

display(IFrame(src=viewer_url, width=1100, height=700))

If you change anything above and want to push it into an already-open viewer
without restarting the server, re-run sections 5 to 7 and then this cell. It
posts the whole scene, camera path included, over the socket.

In [ ]:
import time

time.sleep(5)                  # give the viewer a moment to connect
reader.sendDataViaFlask(port=PORT)
print('scene sent to', viewer_url)

## 9. Recording the movie

In the viewer, under the controls menu:

1. **Camera** pane — tick **Tween** (or press **T** with the render window
   focused) to start the fly-through. It loops, so let it come round to a frame
   you like before recording.
2. **Capture** pane — duration, FPS, filename and format are already set from
   this notebook (`ORBIT_SECONDS` seconds at 30 FPS, PNG). Set **Capture
   Region** to the output resolution you want; frames are rendered at that size,
   not at the window size.
3. Press **Record Video**. A progress circle appears top-right; when it clears,
   the browser saves `brahma_sub<N>_snap<NNN>.tar`, one PNG per frame.

Recording runs in the browser, so the tar lands on whatever machine that browser
is on — if you tunnelled the port from a compute node, look in the *local*
Downloads folder, and run the commands below there too. Firefly's own advice is
to use Firefox: Chrome can stall on long captures.

`-framerate` below must match the FPS you recorded at, or the movie will run at
the wrong speed.

In [ ]:
capture_name = f'brahma_sub{subhalo_index}_snap{SNAP:03d}'
print(f"""
mkdir -p {capture_name}_frames
tar -xf ~/Downloads/{capture_name}.tar -C {capture_name}_frames

ffmpeg -y -framerate 30 -pattern_type glob -i '{capture_name}_frames/*.png' \\
       -c:v libx264 -pix_fmt yuv420p -crf 18 \\
       -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" \\
       {capture_name}.mp4
""")

## 10. Shut the server down

In [ ]:
print('Firefly servers stopped:', quitAllFireflyServers())